In [ ]:
import duckdb
import os

con = duckdb.connect()


query_consolidacion = """
COPY (
    SELECT * 
    FROM read_parquet('../data/bronze/eventos_raw/*.parquet', union_by_name=True)
) TO '../data/silver/eventos_challenger_league.parquet' (FORMAT PARQUET);
"""

con.execute(query_consolidacion)



query_verificacion = """
SELECT 
    event_category, 
    COUNT(*) as total_acciones
FROM '../data/silver/eventos_challenger_league.parquet'
GROUP BY event_category
ORDER BY total_acciones DESC
"""

df_resumen = con.execute(query_verificacion).df()
print("\nResumen de tu nueva Capa Plata optimizada:")
print(df_resumen)

Iniciando procesamiento de Bronce a Plata
Tiempo de procesamiento Bronce a Plata:  0.393648624420166 segundos

Resumen de tu nueva Capa Plata optimizada:
  event_category  total_acciones
0         passes           20323
1   ball-carries            8157
2      defensive            4731
3       dribbles             743


In [ ]:
import duckdb

con = duckdb.connect()


query_enriquecimiento = """
COPY (
    SELECT 
        c.round as matchday,
        c.home_team as home_team,
        c.away_team as away_team,
        l.player_name as player_name,
        l.team_side as team_side,
        l.position as position,
        e.*
    FROM read_parquet('../data/silver/eventos_challenger_league.parquet') e
    LEFT JOIN read_parquet('../data/bronze/lineups.parquet') l
        ON e.match_id = l.match_id AND e.player_id = l.player_id
    LEFT JOIN read_parquet('../data/bronze/calendario_challenger.parquet') c
        ON e.match_id = c.match_id
) TO '../data/silver/eventos_challenger_league_enriched.parquet' (FORMAT PARQUET);
"""

con.execute(query_enriquecimiento)


query_vista = """
SELECT 
    matchday, home_team, player_name, 
    event_category, 
    "playerCoordinates.x" as x_origen, 
    "playerCoordinates.y" as y_origen 
FROM '../data/silver/eventos_challenger_league_enriched.parquet'
WHERE event_category IN ('passes', 'shots', 'duels')
LIMIT 10
"""

df_final = con.execute(query_vista).df()
print("\nAsí luce tu Event Log Enriquecido (Multievento):")
print(df_final)

Iniciando enriquecimiento de la Capa Plata
Tiempo de enriquecimiento de la Capa Plata:  0.062285423278808594 segundos

Así luce tu Event Log Enriquecido (Multievento):
   matchday  home_team     player_name event_category  x_origen  y_origen
0         1  KAS Eupen  Rune Paeshuyse         passes      23.0      49.6
1         1  KAS Eupen  Rune Paeshuyse         passes       7.4      38.0
2         1  KAS Eupen  Rune Paeshuyse         passes      42.3      23.6
3         1  KAS Eupen  Rune Paeshuyse         passes      24.9      29.4
4         1  KAS Eupen  Rune Paeshuyse         passes      26.8      30.1
5         1  KAS Eupen  Rune Paeshuyse         passes      11.6      28.0
6         1  KAS Eupen  Rune Paeshuyse         passes      10.8      16.8
7         1  KAS Eupen  Rune Paeshuyse         passes       8.1      10.6
8         1  KAS Eupen  Rune Paeshuyse         passes      11.0      26.1
9         1  KAS Eupen  Rune Paeshuyse         passes      23.1      27.3


In [ ]:
import duckdb

con = duckdb.connect()
print("Aplicando Normalización Espacial UEFA y Grid xT")

query_espacial = """
COPY (
    SELECT 
        *,
        --Normalización Física a estándar UEFA 105x68m
        ("playerCoordinates.x" / 100.0) * 105.0 AS x_origen_m,
        ("playerCoordinates.y" / 100.0) * 68.0 AS y_origen_m,
        ("passEndCoordinates.x" / 100.0) * 105.0 AS x_destino_m,
        ("passEndCoordinates.y" / 100.0) * 68.0 AS y_destino_m,
        
        -- Normalización Matemática (Cuadrícula xT de 12x8)
        -- Multiplicamos el % por el número de zonas (12 o 8). 
        -- FLOOR para redondear hacia abajo (ej: zona 3.8 -> zona 3).
        -- LEAST(..., 11) o 7 para evitar que un jugador pisando la línea exacta de fondo (100%) asigne una zona fantasma que rompa el modelo.
        LEAST(CAST(FLOOR(("playerCoordinates.x" / 100.0) * 12) AS INTEGER), 11) AS xt_zona_x_origen,
        LEAST(CAST(FLOOR(("playerCoordinates.y" / 100.0) * 8) AS INTEGER), 7) AS xt_zona_y_origen,
        LEAST(CAST(FLOOR(("passEndCoordinates.x" / 100.0) * 12) AS INTEGER), 11) AS xt_zona_x_destino,
        LEAST(CAST(FLOOR(("passEndCoordinates.y" / 100.0) * 8) AS INTEGER), 7) AS xt_zona_y_destino
        
    FROM read_parquet('../data/silver/eventos_challenger_league_enriched.parquet')
) TO '../data/silver/eventos_challenger_league_adjusted.parquet' (FORMAT PARQUET);
"""

con.execute(query_espacial)

query_vista = """
SELECT 
    event_category,
    "playerCoordinates.x" as x_bruto, 
    x_origen_m as x_metros, 
    xt_zona_x_origen as x_grid,
    "playerCoordinates.y" as y_bruto,
    y_origen_m as y_metros,
    xt_zona_y_origen as y_grid
FROM '../data/silver/eventos_challenger_league_adjusted.parquet'
WHERE "playerCoordinates.x" IS NOT NULL
ORDER BY random()
LIMIT 5
"""

df_espacial = con.execute(query_vista).df()
print("\nAsí luce tu normalización (Bruto -> Metros -> Zona xT):")
print(df_espacial)

Aplicando Normalización Espacial UEFA y Grid xT
¡Normalización completada en 0.09 segundos!

Así luce tu normalización (Bruto -> Metros -> Zona xT):
  event_category  x_bruto  x_metros  x_grid  y_bruto  y_metros  y_grid
0         passes     37.2    39.060       4     82.2    55.896       6
1         passes     26.8    28.140       3     67.3    45.764       5
2         passes     74.3    78.015       8     26.5    18.020       2
3   ball-carries     56.3    59.115       6     11.0     7.480       0
4         passes     60.0    63.000       7     15.3    10.404       1


In [ ]:
import duckdb

con = duckdb.connect()

query_shots = """
COPY (
    SELECT
        id as shot_id,
        match_id,
        time as minute,
        "player.id" as player_id,
        "player.name" as player_name,
        "player.position" as player_position,
        isHome as is_home,

        shotType as shot_type,
        situation,
        bodyPart as body_part,

        (100.0 - "playerCoordinates.x") * 1.05 AS x_m,
        "playerCoordinates.y" * 0.68 AS y_m,

        COALESCE(xg, 0.0) AS xg,
        COALESCE(xgot, 0.0) AS xgot,

        FROM read_parquet('../data/bronze/tiros_raw/*.parquet')
        WHERE id IS NOT NULL 
            AND (goalType != 'own' OR goalType IS NULL)
        ) TO '../data/silver/shots_silver.parquet' (FORMAT PARQUET);
"""

con.execute(query_shots)

df_vista = con.execute("""
    SELECT player_name, shot_type, x_m, y_m, xg, xgot 
    FROM '../data/silver/shots_silver.parquet' 
    ORDER BY random() 
    LIMIT 5
""").df()

print("\nMuestra de disparos limpios en Silver:")
print(df_vista)

Iniciando procesamiento de Plata a Oro para Tiros
¡Capa Silver de disparos completada en 0.03 segundos!

Muestra de disparos limpios en Silver:
       player_name shot_type      x_m     y_m        xg      xgot
0      Livio Milts      miss   96.390  20.196  0.036177  0.000000
1   Zakaria Atteri      goal  100.590  22.508  0.101711  0.167492
2   Alliou Dembele      save   89.250  38.760  0.075021  0.106158
3  David Toshevski     block   94.605  32.232  0.079403  0.000000
4      Tom Lockman      miss   78.330  23.528  0.025973  0.000000


In [ ]:
import duckdb

con = duckdb.connect()

query_lineups = """
COPY (
    SELECT 
        l.*,
        CASE 
            WHEN l.team_side = 'home' THEN m.home_team
            WHEN l.team_side = 'away' THEN m.away_team
            ELSE 'unknown'
        END AS team_name
    FROM read_parquet('../data/bronze/lineups.parquet') AS l
    INNER JOIN read_parquet('../data/bronze/calendario_challenger.parquet') AS m
        ON l.match_id = m.match_id
) TO '../data/silver/lineups_silver.parquet' (FORMAT PARQUET);
"""

con.execute(query_lineups)

df = con.execute("""
SELECT player_name, team_side, team_name, position 
FROM '../data/silver/lineups_silver.parquet'
ORDER BY random()
LIMIT 5
""").df()

print("\nMuestra de alineaciones en Silver:")
print(df)


Muestra de alineaciones en Silver:
       player_name team_side          team_name position
0  Lukas Van Eenoo      away  K. Beerschot V.A.        M
1       Isaac Nuhu      home          KAS Eupen        M
2   Nick Gillekens      away        RFC Seraing        G
3   Jordi Palacios      away        KSC Lokeren        D
4      Joel Matadi      away   RSCA Futures U23        D


In [11]:
import duckdb

con = duckdb.connect()

# Inspeccionar el catálogo real de acciones en Silver
acciones_silver = con.execute("""
    SELECT 
        event_category, 
        eventActionType, 
        COUNT(*) AS total_eventos
    FROM read_parquet('../data/gold/challenger_pro_league_xT.parquet')
    GROUP BY event_category, eventActionType
    ORDER BY event_category, total_eventos DESC
""").df()

print(acciones_silver.to_string())

   event_category eventActionType  total_eventos
0    ball-carries      ball-carry           8157
1       defensive   ball-recovery           2106
2       defensive       clearance           1208
3       defensive          tackle            805
4       defensive    interception            459
5       defensive           block            153
6        dribbles         dribble            743
7          passes            pass          19367
8          passes           cross            917
9          passes      ball-touch             26
10         passes            miss             10
11         passes        throw-in              3


In [12]:
con.execute("""
    SELECT 
        event_category, 
        eventActionType, 
        outcome, 
        COUNT(*) AS total
    FROM read_parquet('../data/silver/eventos_challenger_league_adjusted.parquet')
    WHERE eventActionType IN ('tackle', 'dribble')
    GROUP BY event_category, eventActionType, outcome
""").df()

,event_category,eventActionType,outcome,total
0,defensive,tackle,True,536
1,defensive,tackle,<NA>,269
2,dribbles,dribble,True,334
3,dribbles,dribble,<NA>,409
